# 05 · Atendimentos da Vulto: raio-x

Fonte: `atendimentos.csv` (export da ferramenta de atendimento: WhatsApp, Instagram e e-mail, gerado por `padronizar.py`; as 17 linhas duplicadas idênticas já foram removidas).

Objetivo: **entender** os atendimentos, sem decidir nada. A gerente de atendimento quer esses atendimentos **ligados ao cliente e ao pedido certo**.

> Antes de commitar: *Kernel → Restart & Clear Output* (as tabelas têm CPF, e-mail e telefone).

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

In [ ]:
atendimentos = pd.read_csv(STAGING / "atendimentos.csv", dtype=str)
clientes = pd.read_csv(STAGING / "clientes_clientes.csv", dtype=str)
vendas = pd.read_csv(STAGING / "banco_vendas.csv", dtype=str)

## 1. O que é cada linha?

Um atendimento: a conversa inteira com o cliente, com as mensagens do atendente dentro do texto (`[Atendente]: ...`).

In [ ]:
print(atendimentos.shape, "| período:", atendimentos["data_hora"].min()[:10], "a", atendimentos["data_hora"].max()[:10])
print("conversas com resposta do atendente no texto:", atendimentos["mensagem"].fillna("").str.contains(r"\[Atendente\]").sum())
atendimentos.head(3)

## 2. O que significa cada coluna?

In [ ]:
for coluna in ["canal", "status", "assunto", "atendente", "tags", "contato_tipo", "pedido_tipo"]:
    print(coluna, "|", atendimentos[coluna].value_counts(dropna=False).to_dict())

## 3 e 4. Problemas e ligações (cliente e pedido)

In [ ]:
# contato → cliente: telefone e e-mail podem ser procurados no cadastro; Instagram não
telefones, emails = set(clientes["Tel"].dropna()), set(clientes["E-mail"].dropna().str.lower())
def acha(tipo, valor):
    if pd.isna(tipo):
        return "sem contato"
    if tipo == "instagram":
        return "Instagram (sem como achar)"
    return "achou" if (valor in telefones if tipo == "telefone" else valor.lower() in emails) else "não achou"
pd.crosstab(atendimentos["contato_tipo"].fillna("(vazio)"), [acha(t, v) for t, v in zip(atendimentos["contato_tipo"], atendimentos["contato_valor"])])

In [ ]:
# pedido → venda: número do site/app ou número solto (id interno)
codigos = atendimentos["pedido_codigo"].dropna()
print("com pedido:", len(codigos), "| sem pedido:", atendimentos["pedido_codigo"].isna().sum())
print("acham pelo número do site/app:", codigos.isin(vendas["pedido_externo"]).sum(),
      "| acham pelo id interno:", codigos.isin(vendas["id"]).sum(),
      "| não acham:", (~codigos.isin(set(vendas["pedido_externo"].dropna()) | set(vendas["id"]))).sum())
atendimentos.loc[atendimentos["assunto"] == "spam", "mensagem"].str[:60].head()

## Desafios da revisão

In [ ]:
# os números soltos acham a venda certa? a venda deveria ser antes do atendimento, e não muito antes
soltos = atendimentos[atendimentos["pedido_tipo"] == "numero"].dropna(subset=["pedido_codigo"]).merge(
    vendas[["id", "criado_em"]], left_on="pedido_codigo", right_on="id", suffixes=("", "_venda"))
dias = (pd.to_datetime(soltos["data_hora"]) - pd.to_datetime(soltos["criado_em"])).dt.days
print("números soltos que acham uma venda:", len(soltos), "| venda até 60 dias antes do atendimento:", dias.between(0, 60).sum(),
      "| venda depois do atendimento:", (dias < 0).sum(), "| mais de 60 dias antes:", (dias > 60).sum(),
      "| mediana:", dias[dias.between(0, 60)].median(), "dias")

## Anotações para o dicionário de dados

**O que é cada linha**
- Um atendimento (1.647, de 27/02/2025 a 30/09/2026): a conversa inteira, com as respostas do atendente dentro do próprio texto (`[Atendente]: ...`) em 1.105 deles. Não há uma linha por mensagem.

**Colunas**

| Coluna | O que é | Problemas vistos |
|---|---|---|
| `canal` | WhatsApp, Instagram ou e-mail | 9 grafias (`whatsapp`, `WhatsApp`, `wpp`, `insta`, `Instagram DM`...) |
| `contato` / `contato_tipo` / `contato_valor` | quem escreveu e o valor limpo | telefone (794), Instagram (408), e-mail (321), sem contato (124) |
| `assunto` | tema | 17 grafias, 237 vazios; inclui `spam` (10) e `elogio` |
| `status` | situação | 9 grafias: concluído (`ok`, `resolvido`, `Resolvido`, `fechado`, `finalizado`) e em aberto (`em andamento`, `aguardando cliente`, `pendente`, `aberto`); 81 vazios |
| `atendente` | quem atendeu | 8 grafias para ~5 atendentes (`bot` e `Bot Vulto`; `julia` e `Júlia (SAC)`; `Renata` e `renata.sac`) |
| `tags` | etiquetas | 6 combinações (`troca;tamanho`, `entrega;atraso`, `pre-venda`, `vulto+;pontos`...) |
| `pedido` / `pedido_tipo` / `pedido_codigo` | pedido citado e o código limpo | só 678 têm pedido; site `S...`, app `APP-...` ou número solto |

**Ligação com o cliente**
- Telefone: 391 acham o cliente no cadastro, 403 não. E-mail: 198 acham, 123 não.
- **Instagram (408): não há como achar**, porque o cadastro não guarda o @.
- 124 sem contato.

**Ligação com o pedido**
- 678 com pedido: 245 acham pelo número do site/app; 329 por um número solto que bate com o id interno da venda (pode ser coincidência: ligação menos confiável); 104 não acham.

**Problemas dos atendimentos (lista única, nada decidido)**

| # | Problema | Quantos | Afeta |
|---|---|---|---|
| 1 | Instagram sem como ligar ao cliente | 408 | atendimento ligado ao cliente |
| 2 | Telefone ou e-mail que não está no cadastro | 526 | atendimento ligado ao cliente |
| 3 | Sem pedido ou pedido que não acha a venda | 969 + 104 | atendimento ligado ao pedido |
| 4 | Número solto que pode bater com o id errado | 329 | ligação com o pedido |
| 5 | Conversa inteira num campo só | 1.105 | histórico de mensagens no nosso módulo |
| 6 | Canal, status, assunto e atendente sem padrão | todos | de-para para o nosso atendimento |
| 7 | Spam | 10 | o que migrar |

**Desafios da revisão**
- **Os números soltos parecem a venda certa:** dos 329, em 311 a venda é de até 60 dias antes do atendimento (mediana de 11 dias, parecida com os pedidos do site e do app, de 9 dias). Só 18 são suspeitos (11 com a venda depois do atendimento, 7 com mais de 60 dias). A ligação é mais confiável do que parecia; os 18 podem ficar sem pedido.

*(continuar preenchendo ao longo da análise)*